# A toy machine learning model for an example application of FAIR principles to AI models
This toy machine learning model is an example Jupyter notebook to show how it is possible to apply the FAIR principles to AI models at a minimum level. We will use a basic ML model for simplicity purposes. For our example we will build a very simple classifier and use the Iris dataset.

What is a classifier? Roughly, given a collection of records, each record may be seen as a tuple $(x_1,..., x_n, y)$, where $(x_1,..., x_n)$ is the attribute (or feature) set and $y$ is the class label (or target). A classifier is a model that will learn to map each attribute set $(x_1,..., x_n)$ to a class $y$. Thus, given a never-seen-before set of attributes, the model will be able to map it to a class. To learn, this model will use the [k-nearest neighbours algorithm](https://en.wikipedia.org/wiki/K-nearest_neighbors_algorithm).

The Iris dataset contains 5 variables. The first 4 variables are the features of the Iris flower (sepal width and length, and petal width and length), which is the attribute or feature set. The other variable is categorical and refers to the class of the Iris flower (Iris Setosa, Iris Versicolour, or Iris Virginica). 

The Iris dataset is already aligned with the FAIR principles, as it is archived in the UCI Machine Learning repository repository. The dataset is CC-BY so we can use it at the condition of attribution. Here is the full citation: Fisher, R. (1936). Iris [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C56C76. 

Even though the scikit learn ML python packages already loads the data, we will retrieve it directly from the repository for the purpose of this Notebook. The ucimlrepo has a dedicated package to do that (https://github.com/uci-ml-repo/ucimlrepo). Retrieving datasets programmatically from a repository is a very elegant way to be 'Accessible'. Some repositories also allow API. 

Disclaimer: this toy example was partially built with GenAI (Claude, Sonnet 5)

In [ ]:
# import the necessary packages
import seaborn as sns
import pandas as pd # import the pandas package to manipulate the dataset
import sklearn
from sklearn.model_selection import train_test_split # Import train_test_split function
from sklearn import metrics #Import scikit-learn metrics module for accuracy calculation
from sklearn.model_selection import GridSearchCV # for parameter tuning
from sklearn.neighbors import KNeighborsClassifier # this is the algorithm we will use to build our ML model
from sklearn.metrics import ConfusionMatrixDisplay #import the function to see the confusion matrix
from sklearn.metrics import classification_report

# Install the ucimlrepo package 
!pip install ucimlrepo

# Import the dataset into the code
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
iris_fromUCI = fetch_ucirepo(id=53) 

In [ ]:
# We load the dataset as a pandas dataframe
iris_df = iris.data.original
type(iris_df)

In [ ]:
# Let's have a look at the dataset
iris_df.head()

In [ ]:
# The ucimlrepo package also allows you to visualize metadata and variable information

# metadata 
print(iris.metadata) 
  
# variable information 
print(iris.variables) 

## 1) Data exploration
Let's explore the Iris dataset and make sure that the data is clean (e.g., no null values).

In [ ]:
iris_df.info()

In [ ]:
# basic statistics
iris_df.describe()

Let's plot histograms and boxplots to visualize the distribution of data

In [ ]:
hist = iris_df.hist()

In [ ]:
box = sns.boxplot(data = iris_df, orient='h')

## 3) Feature selection
Let us now split the variables of the dataset into two groups:
- X (set of feature attributes, namely the sepal width and length, and petal width and length) and
- y (the target attribute, i.e., the class that our model will learn to predict, namely the type of iris flower).

In [ ]:
X = iris_df[["sepal length", "sepal width",	"petal length", "petal width"]] #features attributes
y = iris_df["class"] #target attributes

## 4) Split the data into train set and test set
For each of our groups, we randomly split the data between training set (X and y) and test set (X and y). We opt for a given random state (here 42) for reproducibility. In our case, test_size = 0.3 means that 30% of the dataset will function as the test set, while the remaining 70% will function as the train set.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
X_train.shape, X_test.shape # we verify the shape of the test and the train (rows, columns)

## 5) K-nearest neighbours (k-NN) classifier
We apply the k-nearest neighbour algorithm to build the classifier model. See [here](https://en.wikipedia.org/wiki/K-nearest_neighbors_algorithm) for more info on the classifier. The k-NN classifier is based on the distance between data points. Thus, we need to first normalize the data, to avoid one variable to prevail over other ones.

In [ ]:
# data standardization
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().set_output(transform="pandas")

scaler.fit(X) # we calculate mean and standard deviation in order to standardize the entire X
# we apply the standardizer to both X_train and X_test
scaled_X_train = scaler.transform(X_train) 
scaled_X_test = scaler.transform(X_test) 

In [ ]:
# we initialize the k-NN classifier object
myKNN_classifier = KNeighborsClassifier()

# we have a look at the parameters of the model
myKNN_classifier.get_params()

In [ ]:
# we tune the parameters using grid search
# we use the grid search just with a few parameters for simplicity
params = {
    'n_neighbors': [2, 3, 4, 5, 6, 7, 8, 9, 10],
    'weights': ['uniform', 'distance']
}
# we set the grid search with cross validation = 5 namely cross validation with 5 fold
GridSearchKNN = GridSearchCV(estimator=myKNN_classifier,
                           param_grid=params,
                           cv=5, n_jobs=-1, verbose=1, scoring = "accuracy")

#we perform the grid search on the training data
GridSearchKNN.fit(scaled_X_train, y_train)

In [ ]:
# select the best classifier and see its parameters
best_knn_classifier = GridSearchKNN.best_estimator_
best_knn_classifier.get_params()

In [ ]:
# fit the best classifier with the data and get confusion matrix
best_knn_classifier = best_knn_classifier.fit(scaled_X_train, y_train)
prediction_of_best_knn_classifier = best_knn_classifier.predict(scaled_X_test)

# I build the confusion matrix
cm = ConfusionMatrixDisplay.from_estimator(
        best_knn_classifier,
        scaled_X_test,
        y_test,
        cmap=plt.cm.Blues
    )
print(cm.confusion_matrix)

In [ ]:
# I get the metrics
print(classification_report(y_test, prediction_of_best_knn_classifier))

In [ ]:
# calculate the f1 score with macro average
sklearn.metrics.f1_score(y_test, prediction_of_best_knn_classifier, average = "macro")